# Simulator

In [ ]:
#|default_exp simulator

In [ ]:
#| export
class Gate:
    def __init__(self, a, b):
        self.a = a; self.b = b
    def __call__(self) -> bool:
        raise NotImplementedError

In [ ]:
#| export
class And(Gate):
    def __call__(self) -> bool:
        return self.a() and self.b()

In [ ]:
#| export
class Input:
    def __init__(self, value: bool):
        self.value = value
    def __call__(self) -> bool:
        return self.value
    def __repr__(self):
        return str(self.value)

In [ ]:
Input(True)

True

In [ ]:
And(Input(True), Input(False))()

False

give me the Or and Xor classes

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

```python
class Or(Gate):
    def __call__(self) -> bool:
        return self.a() or self.b()

class Xor(Gate):
    def __call__(self) -> bool:
        return self.a() ^ self.b()
```

In [ ]:
#| export
class Or(Gate):
    def __call__(self) -> bool:
        return self.a() or self.b()

class Xor(Gate):
    def __call__(self) -> bool:
        return self.a() ^ self.b()


In [ ]:
#| export
class FullAdder:
    def __init__(self, a, b, c_in):
        self.a = a
        self.b = b
        self.c_in = c_in
        self.xor1 = Xor(self.a,self.b)
        self.sum = Xor(self.xor1, self.c_in)
        self.and1 = And(self.a,self.b)
        self.and2 = And(self.xor1, self.c_in)
        self.c_out = Or(self.and1, self.and2)


In [ ]:
adder = FullAdder(Input(True), Input(True), Low)
adder.sum()

NameError: name 'Low' is not defined

In [ ]:
adder.a = Input(False)
adder.sum()

In [ ]:
adder.xor1.a()

In [ ]:
from fastcore.test import test_eq

# Test all 8 possible input combinations for a full adder
# Format: (a, b, c_in) -> (sum, c_out)

# 0 + 0 + 0 = 0, carry 0
adder = FullAdder(Input(False), Input(False), Input(False))
test_eq(adder.sum(), False)
test_eq(adder.c_out(), False)

# 0 + 0 + 1 = 1, carry 0
adder = FullAdder(Input(False), Input(False), Input(True))
test_eq(adder.sum(), True)
test_eq(adder.c_out(), False)

# 0 + 1 + 0 = 1, carry 0
adder = FullAdder(Input(False), Input(True), Input(False))
test_eq(adder.sum(), True)
test_eq(adder.c_out(), False)

# 0 + 1 + 1 = 0, carry 1
adder = FullAdder(Input(False), Input(True), Input(True))
test_eq(adder.sum(), False)
test_eq(adder.c_out(), True)

# 1 + 0 + 0 = 1, carry 0
adder = FullAdder(Input(True), Input(False), Input(False))
test_eq(adder.sum(), True)
test_eq(adder.c_out(), False)

# 1 + 0 + 1 = 0, carry 1
adder = FullAdder(Input(True), Input(False), Input(True))
test_eq(adder.sum(), False)
test_eq(adder.c_out(), True)

# 1 + 1 + 0 = 0, carry 1
adder = FullAdder(Input(True), Input(True), Input(False))
test_eq(adder.sum(), False)
test_eq(adder.c_out(), True)

# 1 + 1 + 1 = 1, carry 1
adder = FullAdder(Input(True), Input(True), Input(True))
test_eq(adder.sum(), True)
test_eq(adder.c_out(), True)

# Test mutability: changing input values propagates through circuit
adder = FullAdder(Input(True), Input(True), Input(False))
test_eq(adder.sum(), False)  # 1 + 1 + 0 = 0 (carry 1)

adder.a.value = False  # Change first input to 0
test_eq(adder.sum(), True)  # Now 0 + 1 + 0 = 1 (carry 0)
test_eq(adder.c_out(), False)


I want to have a representation of the circuit where I get all the gates in a row with on top the value of the pin (to make it easier to compare with a physical IC chip). add using @patch

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Nice idea! So you want something like a visual representation showing gate types and their pin values in a row, similar to reading a physical IC datasheet.

A few quick questions to make sure I understand:

1. Do you want this as a method on `FullAdder` (like `adder.show()`) that displays all its internal gates?

2. For the pin layout, are you thinking something like:
   ```
   a b | out
   1 0 | 0     AND
   ```
   Or a different format?

3. Should it show all gates in the circuit, or just specific ones you care about?

show all the gates, add a method to FullAdder. now I want something nicer looking where things a properly aliged somethinlike this this:
```
High    Low    High
   |          |          |
   a          b        out
         AND 1
```

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Nice! So you want a visual ASCII representation with values on top, pins in the middle, and gate names below, all aligned.

Here's a start for the method structure:

```python
@patch
def show(self:FullAdder):
    # Collect all gates with their inputs and outputs
    # Format and align each gate
    # Print them side by side or stacked
    pass
```

Before I fill this in - do you want:
1. Gates shown **horizontally** (all in one row, side by side)?
2. Or **vertically** (stacked, one gate per section)?

Also, should intermediate wires (like `xor1` output) show their values when they're used as inputs to other gates?

all gates horizontally

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Got it! Here's a method that shows all gates horizontally:

```python
@patch
def show(self:FullAdder):
    gates = [
        ("XOR1", [self.a, self.b], self.xor1),
        ("AND1", [self.a, self.b], self.and1),
        ("XOR2", [self.xor1, self.c_in], self.sum),
        ("AND2", [self.xor1, self.c_in], self.and2),
        ("OR1", [self.and1, self.and2], self.c_out),
    ]
    
    for name, inputs, output in gates:
        vals = [str(int(inp())) for inp in inputs]
        out_val = str(int(output()))
        print(f"{vals[0]:^5} {vals[1]:^5} {out_val:^5}")
        print(f"  |       |       |")
        print(f"  a       b     out")
        print(f"      {name}")
        print()
```

Try it with `adder.show()` and let me know if you want the spacing/alignment adjusted!

In [ ]:
#| export
from fastcore.all import patch, compose, L
from math import ceil, floor

In [ ]:
#| export
@patch
def show(self:FullAdder):
    gates = [
        ("XOR1", self.a, self.b, self.xor1),
        ("AND1", self.a, self.b, self.and1),
        ("XOR2", self.xor1, self.c_in, self.sum),
        ("AND2", self.xor1, self.c_in, self.and2),
        ("OR", self.and1, self.and2, self.c_out),
    ]

    for _, a,b, out in gates:
        print(f"{a()}  {b()}  {out()}", end="\t")
    print()
    for i in range(len(gates)):
        print(" |   |   |", end="\t")
    print()

    for name, *_ in gates:
        print(f"    {name}    ", end="\t")
    print()


In [ ]:
adder.show()

NameError: name 'adder' is not defined

now that this code and fix the aligment, make False and True of the same width

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here's the alignment fix:

```python
@patch
def show(self:FullAdder):
    gates = [
        ("XOR1", self.a, self.b, self.xor1),
        ("AND1", self.a, self.b, self.and1),
        ("XOR2", self.xor1, self.c_in, self.sum),
        ("AND2", self.xor1, self.c_in, self.and2),
        ("OR", self.and1, self.and2, self.c_out),
    ]

    for _, a, b, out in gates:
        print(f"{str(a()):>5}  {str(b()):>5}  {str(out()):>5}", end="\t")
    print()
    for i in range(len(gates)):
        print("  |      |      |  ", end="\t")
    print()
    for name, *_ in gates:
        print(f"      {name}      ", end="\t")
    print()
```

The key change is `str(a()):>5` which right-aligns the boolean strings in a 5-character width. Try it!

In [ ]:
#| export
@patch
def show(self:FullAdder):
    gates = [
        ("XOR1", self.a, self.b, self.xor1),
        ("XOR2", self.xor1, self.c_in, self.sum),
        ("AND1", self.a, self.b, self.and1),
        ("AND2", self.xor1, self.c_in, self.and2),
        ("OR", self.and1, self.and2, self.c_out),
    ]

    sep = " "*5
    for _, a, b, out in gates:
        print(f"{str(a()):>5}  {str(b()):>5}  {str(out()):>5}", end=sep)
    print()
    for i in range(len(gates)):
        print("  |      |      |  ", end=sep)
    print()
    len_spaces = 5*3+2*2 - 4 # 5 is the width of False, 2 times 2 space, 4 is the width of the gate name
    space_before = " " * ceil(len_spaces/2)
    space_after = " " * floor(len_spaces/2)
    for name, *_ in gates:
        print(f"{space_before}{name:^4}{space_after}", end=sep)
    print()


In [ ]:
adder.show()

NameError: name 'adder' is not defined

In [ ]:
L(*"ciao")

In [ ]:
#| export
def dec_to_bin(x):
    """converts decimal to list of 8 bool, from low significance bit"""
    string = '{0:08b}'.format(x)
    return L(*string).map(int).map(bool)[::-1] # first convert string to int, as any non empty string is try

In [ ]:
dec_to_bin(10)

In [ ]:
#| export
class NBitAdder:
    def __init__(self, n_bits: int):
        self.n_bits = n_bits
        self.max_value = 2 ** n_bits
        self.a_bits = [Input(False) for _ in range(n_bits)]
        self.b_bits = [Input(False) for _ in range(n_bits)]
        self.adders = [FullAdder(self.a_bits[0], self.b_bits[0], Input(False))] # first carry is always 0
        for i in range(1, n_bits):
            next_adder = FullAdder(self.a_bits[i], self.b_bits[i], self.adders[i-1].c_out)
            self.adders.append(next_adder)
        self.sum_bits = [adder.sum for adder in self.adders]
    
    def check_input(self, n):
        if not 0 <= n < self.max_value:
            raise ValueError(f"Input too big. A {self.n_bits} n bit adder supports numbers up to {self.max_value}, got {n}")

    def set_inputs(self, a: int, b:int):
        self.check_input(a); self.check_input(b)
        a_bool = dec_to_bin(a)[:self.n_bits]
        b_bool = dec_to_bin(b)[:self.n_bits]
        for i, (a_bit, b_bit) in enumerate(zip(a_bool, b_bool)):
            self.a_bits[i].value = a_bit
            self.b_bits[i].value = b_bit
    
    def sum(self):
        return [sum() for sum in self.sum_bits]
    
    def sum_dec(self):
        bits =  [str(int(x)) for x in self.sum()][::-1] # converts bits to 0/1 chars and the reverse the order
        return int("".join(bits), 2)

In [ ]:
add4 = NBitAdder(4)

In [ ]:
add4.sum()

[False, False, False, False]

In [ ]:
add4.set_inputs(3,5)
add4.a_bits, add4.b_bits, add4.sum(), add4.sum_dec()

NameError: name 'dec_to_bin' is not defined

In [ ]:
from fastcore.test import test_eq, test_fail

# Test 4-bit adder
add4 = NBitAdder(4)

# Basic addition
add4.set_inputs(0, 0); test_eq(add4.sum_dec(), 0)
add4.set_inputs(1, 1); test_eq(add4.sum_dec(), 2)
add4.set_inputs(3, 5); test_eq(add4.sum_dec(), 8)
add4.set_inputs(7, 7); test_eq(add4.sum_dec(), 14)
add4.set_inputs(15, 0); test_eq(add4.sum_dec(), 15)
add4.set_inputs(0, 15); test_eq(add4.sum_dec(), 15)

# Test overflow (wraps around due to carry out being ignored)
add4.set_inputs(15, 1); test_eq(add4.sum_dec(), 0)
add4.set_inputs(15, 15); test_eq(add4.sum_dec(), 14)
add4.set_inputs(8, 8); test_eq(add4.sum_dec(), 0)

# Test input validation
test_fail(lambda: add4.set_inputs(16, 0), contains="too big")
test_fail(lambda: add4.set_inputs(0, 16), contains="too big")

# Test 8-bit adder
add8 = NBitAdder(8)
add8.set_inputs(100, 55); test_eq(add8.sum_dec(), 155)
add8.set_inputs(255, 0); test_eq(add8.sum_dec(), 255)
add8.set_inputs(128, 127); test_eq(add8.sum_dec(), 255)
add8.set_inputs(255, 1); test_eq(add8.sum_dec(), 0)  # overflow

# Test mutability (changing inputs propagates)
add4.set_inputs(3, 3); test_eq(add4.sum_dec(), 6)
add4.a_bits[0].value = False; test_eq(add4.sum_dec(), 5)  # changed 3 to 2, so 2+3=5

# Exhaustive test for 4-bit (all 256 combinations)
add4 = NBitAdder(4)
for a in range(16):
    for b in range(16):
        add4.set_inputs(a, b)
        test_eq(add4.sum_dec(), (a + b) % 16)


NameError: name 'dec_to_bin' is not defined

add a show method to the NBitAdder class

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

I can help with that! A few options for how to display the n-bit adder:

1. **Show all full adders horizontally** (like the `FullAdder.show()` but repeated for each bit)
2. **Show a compact summary** with just the input bits, output bits, and carry chain
3. **Show bit-by-bit vertically** (one full adder per row)

Which style would you prefer? Or do you have a different visualization in mind?

for now I want all the adders from FullAdder.show stacked

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Got it! So you want to call `show()` on each full adder in sequence, stacking them vertically.

Here's a simple version:

```python
@patch
def show(self:NBitAdder):
    for i, adder in enumerate(self.adders):
        print(f"Bit {i}:")
        adder.show()
        print()
```

Want to try that, or would you like the bit labels formatted differently?

In [ ]:
#| export
@patch
def show(self:NBitAdder):
    print(f"Result: {self.sum_dec()} {self.sum()}")
    for i, adder in enumerate(self.adders):
        print(f"Bit {i}:")
        adder.show()
        print()


In [ ]:
add4.set_inputs(3,5)
add4.show()

Result: 8 [False, False, False, True]
Bit 0:
 True   True  False     False  False  False      True   True   True     False  False  False      True  False   True     
  |      |      |         |      |      |         |      |      |         |      |      |         |      |      |       
        XOR1                    XOR2                    AND1                    AND2                     OR             

Bit 1:
 True  False   True      True   True  False      True  False  False      True   True   True     False   True   True     
  |      |      |         |      |      |         |      |      |         |      |      |         |      |      |       
        XOR1                    XOR2                    AND1                    AND2                     OR             

Bit 2:
False   True   True      True   True  False     False   True  False      True   True   True     False   True   True     
  |      |      |         |      |      |         |      |      |         |      |      |   

# Debug check

the board is accessible at 8bit-adder.mone27.net and numbers can be adder using 8bit-adder. mone27.net/set_a/{number} 


give me a set_a and set_b functions that work over the API call

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

```python
import httpx

def set_a(n:int): httpx.get(f"http://8bit-adder.mone27.net/set_a/{n}")
def set_b(n:int): httpx.get(f"http://8bit-adder.mone27.net/set_b/{n}")
```

In [ ]:
import httpx

def set_a(n:int): httpx.get(f"http://8bit-adder.mone27.net/set_a/{n}")
def set_b(n:int): httpx.get(f"http://8bit-adder.mone27.net/set_b/{n}")


In [ ]:
set_a(10)
set_b(5)

In [ ]:
#| export
set_b(3)
set_a(12)

In [ ]:
def get_expected(a, b):
    set_a(a)
    set_b(b)
    res = a + b
    print(f"{res:04b} = {a:04b} + {b:04b} ({a} + {b} = {res})")
    return res
def get_actual():
    n = input("actual:")
    return int(n, 2) # to decimal

In [ ]:
def test_res(a, b, n_bits=4):
    res = get_expected(a, b)
    actual = get_actual()
    max_val = 2 ** n_bits
    res_masked = res % max_val
    actual_masked = actual % max_val
    if res == actual:
        print("✅ Success\n")
    elif res_masked == actual_masked:
        print(f"⚠️  Higher bits wrong: got {actual:04b}, expected {res:04b} (lower {n_bits} bits correct)\n")
    else:
        print(f"❌ Failure: got {actual:04b} != {res:04b}\n")
    return res, actual


In [ ]:
def test_2_bits():
    for a in range(4):
        for b in range(4):
            test_res(a, b)

In [ ]:
def test_n_bit(n_bit):
    shift = n_bit - 2 # test all but the least 2 significant bits
    for a in range(4):
        for b in range(4):
            test_res(a << shift, b << shift, n_bits=n_bit)

In [ ]:
test_n_bit(2)

0000 = 0000 + 0000 (0 + 0 = 0)


✅ Success



0001 = 0000 + 0001 (0 + 1 = 1)


✅ Success



0010 = 0000 + 0010 (0 + 2 = 2)


...

✅ Success



0101 = 0011 + 0010 (3 + 2 = 5)


✅ Success



0110 = 0011 + 0011 (3 + 3 = 6)


✅ Success



In [ ]:
test_n_bit(3)

0000 = 0000 + 0000 (0 + 0 = 0)


✅ Success



0010 = 0000 + 0010 (0 + 2 = 2)


✅ Success



0100 = 0000 + 0100 (0 + 4 = 4)


...

✅ Success



1010 = 0110 + 0100 (6 + 4 = 10)


⚠️  Higher bits wrong: got 0010, expected 1010 (lower 3 bits correct)



1100 = 0110 + 0110 (6 + 6 = 12)


❌ Failure: got 0000 != 1100



In [ ]:
test_n_bit(4)

0000 = 0000 + 0000 (0 + 0 = 0)


✅ Success



0100 = 0000 + 0100 (0 + 4 = 4)


❌ Failure: got 1100 != 0100



1000 = 0000 + 1000 (0 + 8 = 8)


...

❌ Failure: got 1000 != 10000



10100 = 1100 + 1000 (12 + 8 = 20)


⚠️  Higher bits wrong: got 0100, expected 10100 (lower 4 bits correct)



11000 = 1100 + 1100 (12 + 12 = 24)


❌ Failure: got 0000 != 11000



In [ ]:
test_res(0,4)

0100 = 0000 + 0100 (0 + 4 = 4)


❌ Failure: got 1100 != 0100



(4, 12)

In [ ]:
add4 = NBitAdder(4)
add4.set_inputs(0, 4)
set_a(0); set_b(4)
add4.show()


Bit 0:
False  False  False     False  False  False     False  False  False     False  False  False     False  False  False     
  |      |      |         |      |      |         |      |      |         |      |      |         |      |      |       
        XOR1                    XOR2                    AND1                    AND2                     OR             

Bit 1:
False  False  False     False  False  False     False  False  False     False  False  False     False  False  False     
  |      |      |         |      |      |         |      |      |         |      |      |         |      |      |       
        XOR1                    XOR2                    AND1                    AND2                     OR             

Bit 2:
False   True   True      True  False   True     False   True  False      True  False  False     False  False  False     
  |      |      |         |      |      |         |      |      |         |      |      |         |      |      |       
        X

the input b for the 4th should be low but is high .... I guess is the inverter fault, will have a look at it

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good catch! So your simulation shows the correct state (bit 2 of `b` is True for value 4, bits 0,1,3 are False), but your physical circuit has bit 3 stuck high. An inverter issue would definitely explain that kind of behavior — either it's inverted when it shouldn't be, or there's a wiring issue feeding into it.

Let me know what you find, or if you'd like to add any debugging helpers to the simulator!

In [ ]:
add3 = NBitAdder(3)
add3.set_inputs(2, 6)
add3.show()


Bit 0:
False  False  False     False  False  False     False  False  False     False  False  False     False  False  False     
  |      |      |         |      |      |         |      |      |         |      |      |         |      |      |       
        XOR1                    XOR2                    AND1                    AND2                     OR             

Bit 1:
 True   True  False     False  False  False      True   True   True     False  False  False      True  False   True     
  |      |      |         |      |      |         |      |      |         |      |      |         |      |      |       
        XOR1                    XOR2                    AND1                    AND2                     OR             

Bit 2:
False   True   True      True   True  False     False   True  False      True   True   True     False   True   True     
  |      |      |         |      |      |         |      |      |         |      |      |         |      |      |       
        X